In [17]:
pip install PyPDF2 python-docx sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [20]:
import fitz

print("PyMuPDF loaded successfully")

PyMuPDF loaded successfully


In [18]:
import os
from PyPDF2 import PdfReader
import docx
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [19]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

Embedding model loaded


In [21]:
def read_pdf(path):
    reader = PdfReader(path)
    text = ""
    
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    
    return text

In [22]:
def read_docx(path):
    doc = Document(path)
    text = ""
    
    for para in doc.paragraphs:
        text += para.text + "\n"
    
    return text

In [23]:
import os
def load_documents(folder="project"):

    text = ""
    file_count = 0

    for file in os.listdir(folder):

        path = os.path.join(folder, file)

        if file.endswith(".pdf"):
            print("Reading PDF:", file)
            text += read_pdf(path)
            file_count += 1

        elif file.endswith(".docx"):
            print("Reading DOCX:", file)
            text += read_docx(path)
            file_count += 1

    print("Total files processed:", file_count)

    return text

In [24]:
text = load_documents("project")

print("\nTotal characters extracted:", len(text))
print("\nPreview of extracted text:\n")
print(text[:1000])

Reading PDF: board-model-risk-management-SABR-BETR-models-dec2022.pdf
Reading DOCX: model_documentation_sample.docx
Total files processed: 2

Total characters extracted: 51921

Preview of extracted text:

  
2022 -SR-B-016 1 of 30 
Evaluation Report  
2022 -SR-B-016 
December 7, 2022  
Board of Governors of the Federal Reserve System  
The Board Can Enhance the Effectiveness 
of Certain Aspects of Its Model Risk 
Management Processes for the  
SR/HC -SABR and BETR Models  
  
2022 -SR-B-016 2 of 30 Executive Summary, 2022 -SR-B-016, December 7, 2022  
The Board Can Enhance the Effectiveness of Certain Aspects of Its 
Model Risk Management Processes for the SR/HC -SABR and BETR 
Models  
Findings  
The Division of Supervision and Regulation (S&R) uses supervisory 
models , such as the Supervision and Regulation  Statistical 
Assessment of Bank Risk ( SR-SABR) , Holding Company Statistical 
Assessment of Bank Risk  (HC-SABR)  (together, SR/HC -SABR),  and 
Bank Exams Tailored to Risk (BE

In [25]:
def section_chunking(text):

    pattern = r'(Section\s\d+|\d+\.\d+|\d+\.\d+\.\d+)'

    splits = re.split(pattern, text)

    chunks = []

    for i in range(1, len(splits)-1, 2):

        title = splits[i]
        content = splits[i+1]

        chunks.append(title + "\n" + content.strip())

    return chunks

In [26]:
def embed_chunks(chunks):

    embeddings = embedding_model.encode(chunks)

    return np.array(embeddings)

In [27]:
def create_index(embeddings):

    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)

    index.add(embeddings)

    return index

In [28]:
def search(query, index, chunks, k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = []

    for i in indices[0]:
        results.append(chunks[i])

    return results

In [42]:
!ollama pull llama3

]11;?\pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [29]:
def ask_ollama(context, question):

    prompt = f"""
You are a regulatory model risk expert.

Answer using ONLY the context below.

Context:
{context}

Question:
{question}
"""

    response = ollama.chat(
        model="llama3",
        messages=[{"role":"user","content":prompt}]
    )

    return response["message"]["content"]

In [30]:
def build_rag():

    print("Loading documents...")

    text = load_documents()

    print("Creating section chunks...")

    chunks = section_chunking(text)

    print("Total chunks:", len(chunks))

    print("Creating embeddings...")

    embeddings = embed_chunks(chunks)

    print("Building FAISS index...")

    index = create_index(embeddings)

    return index, chunks

In [31]:
index, chunks = build_rag()

Loading documents...
Reading PDF: board-model-risk-management-SABR-BETR-models-dec2022.pdf
Reading DOCX: model_documentation_sample.docx
Total files processed: 2
Creating section chunks...
Total chunks: 4
Creating embeddings...
Building FAISS index...


In [33]:
question = "What is the model ID"

relevant = search(question, index, chunks)

context = "\n".join(relevant)

answer = ask_ollama(context, question)

print(answer)

Since there is no specific mention of a model name in the provided context, I would say that the model does not have a specific name mentioned. It appears to be a Probability of Default (PD) model used for retail banking customers.
